# R9 Kármán-prior diagnostic — no training

This notebook tests the **existing R9 tap-derived von Kármán street prior** against the real
Re=100 CFD data before changing the ModalPINN again.

It does **not** train a network.

### Questions answered

1. How close is the pressure-derived analytical \(k=1\) street to the real CFD \(k=1\) mode?
2. If we allow only one optimal complex amplitude/phase correction ("oracle scale"), how much better can it get?
3. Does the existing R9 componentwise complex trust box still allow the dead wake \(q_1=0\)?
4. Does that same trust box actually contain the true CFD \(k=1\) mode?
5. How does a genuinely radial complex trust region compare?
6. Does a soft anchored Kármán prior give a non-zero gradient at the collapsed wake?
7. Optional: does the analytical street resemble the CFD **vorticity** mode, not just velocity?

### Fairness rule

The analytical street parameters are the same R9 parameters derived from the **32 pressure taps + classical relations**.
CFD full-field data are used only for **diagnosis/evaluation**, never to construct the pressure-only prior.

The notebook automatically uses the `street_prior_used.npz` saved by the completed R9 run if it can find it.
If it cannot, it regenerates the prior using the **exact `street_prior.py` from the R9 notebook**.

## 1. Mount Drive and locate the existing dataset / R9 prior

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob, shutil, json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DRIVE_DATA = '/content/drive/MyDrive/ModalPINN_data/fixed_cylinder_atRe100'
LOCAL_DATA = '/content/fixed_cylinder_atRe100'
RESULT_ROOT = '/content/drive/MyDrive/ModalPINN_results'
OUT_DIR = os.path.join(RESULT_ROOT, 'R9_prior_diagnostic')
os.makedirs(OUT_DIR, exist_ok=True)

assert os.path.exists(DRIVE_DATA), f'Dataset not found: {DRIVE_DATA}'

if not os.path.exists(LOCAL_DATA) or os.path.getsize(LOCAL_DATA) != os.path.getsize(DRIVE_DATA):
    print('Copying the ~1.2 GB CFD file from Drive to local Colab storage...')
    shutil.copyfile(DRIVE_DATA, LOCAL_DATA)
else:
    print('Local CFD copy already exists.')

# Prefer the prior actually saved with the completed production R9 run.
prior_candidates = sorted(
    glob.glob(os.path.join(RESULT_ROOT, 'R9_TRUST_street_*', 'street_prior_used.npz')),
    key=os.path.getmtime
)
if prior_candidates:
    PRIOR_PATH = prior_candidates[-1]
    print('Using completed-R9 prior:', PRIOR_PATH)
else:
    PRIOR_PATH = '/content/street_prior_Ntap32.npz'
    print('No saved R9 prior auto-found. The fallback cell below will regenerate it.')

print('CFD:', LOCAL_DATA, os.path.getsize(LOCAL_DATA), 'bytes')
print('Outputs will be saved to:', OUT_DIR)

## 2. Exact R9 data reader

This is copied from the R9 notebook so we are not changing the data interpretation.

In [ ]:
%%writefile /content/text_flow.py
"""

Author: Mouad Boudina
From: https://zenodo.org/record/5039610


The flow file structure is the following:

Re Ur
(blank line)
Nt N_nodes (Nt = length of the timeline of the flow simulation)
(blank line)
t0
node0_x node0_y U(node0) V(node0) p(node0)
node1_x node1_y U(node1) V(node1) p(node1)
...
t1
node0_x node0_y U(node0) V(node0) p(node0)
node1_x node1_y U(node1) V(node1) p(node1)
...

"""
import time
import numpy as np
#==============================================================================

def floatIt(l):
    return np.array([float(e) for e in l])

def intIt(l):
    return np.array([int(e) for e in l])

def read_flow(infile):
    f = open(infile, 'r')

    t1 = time.process_time()

    print('Reading flow...')

    Re, Ur = floatIt(f.readline().strip().split())

    f.readline() # blank line

    Nt, N_nodes = intIt(f.readline().strip().split())

    f.readline()

    times = []

    nodes_X, nodes_Y = [], []
    Us, Vs, ps = [], [], []

    for n in range(Nt):
        tn = float(f.readline().strip())
        times.append(tn)

        print('%.3f' % tn)

        tmp_nodes_X, tmp_nodes_Y = [], []
        tmp_Us, tmp_Vs, tmp_ps = [], [], []

        for k in range(N_nodes):
            x, y, U, V, p = floatIt(f.readline().strip().split())

            tmp_nodes_X.append(x)
            tmp_nodes_Y.append(y)

            tmp_Us.append(U)
            tmp_Vs.append(V)
            tmp_ps.append(p)

        nodes_X.append(tmp_nodes_X)
        nodes_Y.append(tmp_nodes_Y)

        Us.append(tmp_Us)
        Vs.append(tmp_Vs)
        ps.append(tmp_ps)

    cpu_time = time.process_time() - t1
    print('Done!')
    print('CPU_TIME = %f seconds' % cpu_time)

    f.close()

    return Re, Ur, np.array(times), \
           np.array(nodes_X), np.array(nodes_Y), \
           np.array(Us), np.array(Vs), np.array(ps)

def write_flow(flow, outfile):
    f = open(outfile, 'w')

    t1 = time.process_time()

    print('Writing flow...')

    f.write('%.0f %.1f\n' % (flow.Re, flow.Ur))
    f.write('\n') # blank line

    Nt, N_nodes = len(flow.times), len(flow.nodes_X[0])

    f.write('%d %d\n' % (Nt, N_nodes))
    f.write('\n')

    for n in range(Nt):
        tn = flow.times[n]

        print('%.6f' % tn)

        f.write('%.6f\n' % tn)

        for k in range(N_nodes):
            f.write('%13.9f %13.9f %13.9f %13.9f %13.9f\n' %\
                    (flow.nodes_X[n, k],
                     flow.nodes_Y[n, k],
                     flow.Us[n, k],
                     flow.Vs[n, k],
                     flow.ps[n, k]))

    cpu_time = time.process_time() - t1
    print('Done!')
    print('CPU_TIME = %f seconds' % cpu_time)

    f.close()

## 3. Exact R9 street-prior generator

This is also copied directly from the completed R9 notebook. It is used only as a fallback if
`street_prior_used.npz` cannot be found in the R9 result folder.

In [ ]:
%%writefile /content/street_prior.py
"""R9: derive the closed-form vortex-street prior from the 32 wall taps.

Standalone, numpy-only (no TF). Run BEFORE training:

    python street_prior.py --DataFile Data/fixed_cylinder_atRe100 --NTaps 32

Writes street_prior_Ntap<N>.npz with the closed-form street parameters that
ModalPINN_VortexShedding.py --TrustStreet consumes.

EVERY number here derives from the tap pressures + classical physics:
- omega0: nonlinear sinusoid fit to the tap-integrated lift series
- Gamma:  von Karman drag relation == tap-measured pressure drag / 0.75
          (0.75 = pressure share of total drag at Re~100, textbook value)
- Uc, a:  self-consistent street kinematics (Uc = 1 - Gamma/(sqrt8 a),
          a = 2 pi Uc / omega0)
- xf, r0, phase: matching the IMAGE-SYSTEM street's induced k=1 surface
          pressure pattern to the measured tap k=1 harmonics (the
          Milne-Thomson images make the surface pattern orientation-aware)
- closed-form calibration: the TF-portable single-harmonic expansion is
          aligned (phase offset + amplitude scale) against the numeric
          Lamb-Oseen street ON WAKE PROBE POINTS - a street-to-street
          calibration, no reference data involved.

The reference CFD file is read ONLY to extract the tap pressures - the
exact signals the training script itself trains on.

Method developed and validated in R9_wake_rescue/ (see REPORT.md there).
"""
import argparse
import os

import numpy as np
from scipy.optimize import least_squares

from text_flow import read_flow

# geometry, matching ModalPINN_VortexShedding.py
X_C, Y_C, R_C = 0.0, 0.0, 0.5
LXMIN, LXMAX, LYMIN, LYMAX = -4.0, 8.0, -4.0, 4.0
GEOM = [LXMIN, LXMAX, LYMIN, LYMAX, X_C, Y_C, R_C]
D = 2 * R_C
RE = 100.0
NU = 1.0 / RE
HA_RATIO = 0.281


# ===========================================================================
# numeric street (Lamb-Oseen rows + Milne-Thomson images + dipole)
# - reference implementation for the fit; identical math to
#   R9_wake_rescue/src/analytic_street.py
# ===========================================================================
class Street:
    def __init__(self, Gamma, U_c, x_f=1.0, r0=0.3, phase=0.0, omega=1.036,
                 ramp=0.75):
        self.G, self.Uc, self.omega = Gamma, U_c, omega
        self.a = 2 * np.pi * U_c / omega
        self.h = HA_RATIO * self.a
        self.xf, self.r0, self.phase, self.ramp = x_f, r0, phase, ramp

    def _vortex_positions(self, t, nwin=30):
        ks = np.arange(-nwin, nwin + 1)
        shift = (self.Uc * t + self.phase / self.omega * self.Uc) % self.a
        xu = self.xf + self.a * ks + shift
        xl = self.xf + self.a * (ks + 0.5) + shift
        return (np.stack([xu, np.full_like(xu, +self.h / 2)], 1),
                np.stack([xl, np.full_like(xl, -self.h / 2)], 1))

    def _induced(self, pts, vort_xy, gamma, core_from_x=None):
        dx = pts[:, None, 0] - vort_xy[None, :, 0]
        dy = pts[:, None, 1] - vort_xy[None, :, 1]
        r2 = dx ** 2 + dy ** 2 + 1e-12
        xv = vort_xy[:, 0] if core_from_x is None else core_from_x
        rc2 = self.r0 ** 2 + 4 * NU * np.clip(xv - self.xf, 0, None) / self.Uc
        fac = (1 - np.exp(-r2 / rc2[None, :])) / (2 * np.pi * r2)
        return (-gamma * dy * fac).sum(1), (gamma * dx * fac).sum(1)

    def velocity(self, pts, t):
        up, lo = self._vortex_positions(t)
        uu, vu = self._induced(pts, up, -self.G)
        ul, vl = self._induced(pts, lo, +self.G)
        u, v = uu + ul, vu + vl
        a2 = R_C ** 2
        for row, g in ((up, -self.G), (lo, +self.G)):
            r2v = row[:, 0] ** 2 + row[:, 1] ** 2
            img = row * (a2 / r2v)[:, None]
            ui, vi = self._induced(pts, img, -g, core_from_x=row[:, 0])
            u, v = u + ui, v + vi
        env = 0.5 * (1 + np.tanh((pts[:, 0] - self.xf) / self.ramp))
        x, y = pts[:, 0], pts[:, 1]
        r2 = x ** 2 + y ** 2 + 1e-12
        u_mean = 1.0 - a2 * (x ** 2 - y ** 2) / r2 ** 2
        v_mean = -a2 * 2 * x * y / r2 ** 2
        return u_mean + u * env, v_mean + v * env

    def pressure(self, pts, t):
        u, v = self.velocity(pts, t)
        return -0.5 * ((u - self.Uc) ** 2 + v ** 2)

    def modes(self, pts, nk=3, nt=16):
        T = 2 * np.pi / self.omega
        ts = np.arange(nt) * T / nt
        U = np.empty((nt, len(pts))); V = np.empty_like(U); P = np.empty_like(U)
        for i, t in enumerate(ts):
            U[i], V[i] = self.velocity(pts, t)
            P[i] = self.pressure(pts, t)
        out = {}
        for name, F in (('u', U), ('v', V), ('p', P)):
            c = np.fft.fft(F, axis=0) / nt
            out[name] = [c[0].real] + [c[k] for k in range(1, nk + 1)]
        return out


def uc_of_gamma(G, omega):
    Uc = 0.85
    for _ in range(50):
        a = 2 * np.pi * Uc / omega
        Uc_new = 1.0 - G / (np.sqrt(8.0) * a)
        if abs(Uc_new - Uc) < 1e-12:
            break
        Uc = Uc_new
    return Uc, 2 * np.pi * Uc / omega


def karman_drag_CD(G, Uc, a, h):
    u_ind = G / (np.sqrt(8.0) * a)
    return ((G * h / a) * (1.0 - 2.0 * u_ind) + G ** 2 / (2 * np.pi * a)) \
        / (0.5 * D)


# ===========================================================================
# closed-form street (TF-portable) - identical math to
# R9_wake_rescue/src/closed_form_street.py, numpy backend
# ===========================================================================
def cf_modes_uv(x, y, prm, nk=3):
    """One-sided modes k=1..nk of the closed-form street. Returns us, vs."""
    G, Uc, xf, r0, phase, omega, ramp, delta = (
        prm['Gamma'], prm['Uc'], prm['xf'], prm['r0'], prm['phase'],
        prm['omega'], prm.get('ramp', 0.75), prm.get('delta', 0.35))
    a = 2 * np.pi * Uc / omega
    h = HA_RATIO * a
    env = 0.5 * (1 + np.tanh((x - xf) / ramp))
    rc2 = r0 ** 2 + 4 * NU * np.clip(x - xf, 0, None) / Uc
    us, vs = [], []
    for k in range(1, nk + 1):
        att = np.exp(-(np.pi * k) ** 2 * rc2 / a ** 2)
        tot_u = np.zeros_like(x, dtype=complex)
        tot_v = np.zeros_like(x, dtype=complex)
        for y_row, sgn_row, x0 in ((+h / 2, -1.0, xf), (-h / 2, +1.0, xf + a / 2)):
            yp = y - y_row
            sabs = np.sqrt(yp ** 2 + delta ** 2) - delta
            sgn = -np.tanh(yp / delta)
            Dk = np.exp(-2 * np.pi * k * sabs / a)
            ph = -2 * np.pi * k * (x - x0) / a - k * phase
            Ek = np.cos(ph) + 1j * np.sin(ph)
            base = sgn_row * G / (2 * a) * Ek * Dk * att
            tot_u = tot_u + sgn * base
            tot_v = tot_v + 1j * base
        us.append(tot_u * env)
        vs.append(tot_v * env)
    return us, vs


# ===========================================================================
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--DataFile', default='Data/fixed_cylinder_atRe100')
    ap.add_argument('--NTaps', type=int, default=32)
    ap.add_argument('--Out', default=None)
    args = ap.parse_args()
    out_path = args.Out or f'street_prior_Ntap{args.NTaps}.npz'

    # ---- 1. tap pressures - same selection logic as Load_train_data_desync
    # cut_simu_cylinder_only (transcribed, not imported: that module imports
    # tensorflow, which this numpy-only script must not depend on).
    Re_, Ur_, times, nodes_X, nodes_Y, Us, Vs, Ps = read_flow(args.DataFile)
    eps = 1e-5
    r_all = np.sqrt((nodes_X[0, :] - X_C) ** 2 + (nodes_Y[0, :] - Y_C) ** 2)
    idx_cyl = np.argwhere((r_all - R_C) ** 2 < eps)[:, 0]
    xc_all, yc_all = nodes_X[0, idx_cyl], nodes_Y[0, idx_cyl]
    s_lin = np.linspace(0., 1., args.NTaps, endpoint=False)
    x_t = X_C + R_C * np.cos(2 * np.pi * s_lin)
    y_t = Y_C + R_C * np.sin(2 * np.pi * s_lin)
    pick = np.array([np.argmin((xc_all - x_t[k]) ** 2 + (yc_all - y_t[k]) ** 2)
                     for k in range(args.NTaps)])
    print('Cylinder taps requested: %d, distinct mesh nodes found: %d'
          % (args.NTaps, len(np.unique(pick))))
    x_cyl, y_cyl = xc_all[pick], yc_all[pick]
    p_cyl = Ps[:, idx_cyl[pick]]             # (Nt, NTaps)
    t = np.asarray(times) - times[0]
    print(f'taps: {p_cyl.shape}, t in [0, {t[-1]:.1f}]')

    theta = np.arctan2(y_cyl - Y_C, x_cyl - X_C)
    order = np.argsort(theta)
    th_s, p_s = theta[order], p_cyl[:, order]
    dth = np.diff(np.concatenate([th_s, [th_s[0] + 2 * np.pi]]))
    w = 0.5 * (dth + np.roll(dth, 1))
    CD = -(p_s * np.cos(th_s)[None, :] * w[None, :]).sum(1) * R_C / (0.5 * D)
    CL = -(p_s * np.sin(th_s)[None, :] * w[None, :]).sum(1) * R_C / (0.5 * D)

    # ---- 2. omega0 from a nonlinear sinusoid fit to CL
    z = CL - CL.mean()
    crossings = np.where(np.diff(np.sign(z)) != 0)[0]
    w_init = np.pi / np.mean(np.diff(t[crossings]))
    fit = least_squares(
        lambda prm: prm[0] * np.sin(prm[2] * t + prm[1]) + prm[3] - CL,
        [0.5 * (CL.max() - CL.min()), 0.0, w_init, CL.mean()], method='lm')
    omega = abs(float(fit.x[2]))
    CD0 = float(CD.mean())
    print(f'omega0_hat = {omega:.5f}  CD_pressure = {CD0:.4f}')

    # ---- 3. per-tap k=1 harmonics
    cols = [np.ones_like(t), np.cos(omega * t), np.sin(omega * t)]
    A = np.stack(cols, 1)
    cf_, *_ = np.linalg.lstsq(A, p_s, rcond=None)
    p1_meas = 0.5 * (cf_[1] - 1j * cf_[2])

    # ---- 4. Gamma from the Karman drag relation (bisection)
    CD_target = CD0 / 0.75
    lo, hi = 0.5, 6.0
    for _ in range(60):
        G = 0.5 * (lo + hi)
        Uc, a = uc_of_gamma(G, omega)
        if karman_drag_CD(G, Uc, a, HA_RATIO * a) < CD_target:
            lo = G
        else:
            hi = G
    G = 0.5 * (lo + hi)
    Uc, a = uc_of_gamma(G, omega)
    print(f'Gamma = {G:.3f}  Uc = {Uc:.3f}  a = {a:.3f}')

    # ---- 5. xf, r0, phase from the tap k=1 pattern (image street)
    tap_pts = np.stack([R_C * np.cos(th_s), R_C * np.sin(th_s)], 1)
    best = None
    for xf in (0.6, 0.8, 1.0, 1.2):
        for r0 in (0.2, 0.3, 0.4):
            st = Street(G, Uc, x_f=xf, r0=r0, omega=omega)
            sm = st.modes(tap_pts, nk=1, nt=16)
            p1s = sm['p'][1]
            corr = np.abs(np.vdot(p1s, p1_meas)) / (
                np.linalg.norm(p1s) * np.linalg.norm(p1_meas))
            phi = np.angle(np.vdot(p1s, p1_meas))
            if best is None or corr > best[0]:
                best = (corr, xf, r0, phi)
    corr_tap, xf, r0, phi = best
    print(f'xf = {xf}  r0 = {r0}  phase = {phi:+.3f}  tap-p1 corr = {corr_tap:.3f}')

    # ---- 6. calibrate the closed form against the numeric street
    num = Street(G, Uc, x_f=xf, r0=r0, phase=phi, omega=omega)
    rng = np.random.default_rng(3)
    pts = rng.uniform([1.0, -2.0], [8.0, 2.0], size=(1500, 2))
    sm = num.modes(pts, nk=3, nt=16)
    prm = dict(Gamma=G, Uc=Uc, xf=xf, r0=r0, omega=omega, phase=phi)
    best = None
    for extra in np.linspace(-np.pi, np.pi, 48, endpoint=False):
        prm['phase'] = phi + extra
        us, vs = cf_modes_uv(pts[:, 0], pts[:, 1], prm, nk=1)
        inner = np.vdot(vs[0], sm['v'][1])
        corr = abs(inner) / (np.linalg.norm(vs[0])
                             * np.linalg.norm(sm['v'][1]) + 1e-30)
        score = corr - abs(np.angle(inner)) * 0.05
        if best is None or score > best[0]:
            best = (score, corr, extra)
    _, corr_cf, extra = best
    prm['phase'] = phi + extra
    us, vs = cf_modes_uv(pts[:, 0], pts[:, 1], prm, nk=1)
    amp_scale = float(np.linalg.norm(sm['v'][1]) / np.linalg.norm(vs[0]))
    # pressure anchor: p_k ~ -(1-Uc) u_k, amplitude-matched at k=1
    p1_approx = -(1.0 - Uc) * us[0] * amp_scale
    scale_p = float(np.linalg.norm(sm['p'][1]) / np.linalg.norm(p1_approx))
    print(f'closed-form calibration: corr vs numeric = {corr_cf:.3f}, '
          f'amp_scale = {amp_scale:.3f}, scale_p = {scale_p:.3f}')
    assert corr_cf > 0.95, 'closed-form street failed to match numeric street'

    np.savez(out_path,
             Gamma=G, Uc=Uc, xf=xf, r0=r0, omega=omega,
             phase=prm['phase'], amp_scale=amp_scale, scale_p=scale_p,
             ramp=0.75, delta=0.35,
             CD_pressure=CD0, tap_p1_corr=corr_tap,
             cf_corr_vs_numeric=corr_cf)
    print('saved', out_path)


if __name__ == '__main__':
    main()

## 4. Load (or regenerate) the exact R9 prior

In [ ]:
if not os.path.exists(PRIOR_PATH):
    print('Regenerating the exact R9 32-tap prior...')
    import subprocess
    cmd = ['python', '/content/street_prior.py',
           '--DataFile', LOCAL_DATA, '--NTaps', '32',
           '--Out', '/content/street_prior_Ntap32.npz']
    subprocess.run(cmd, check=True)
    PRIOR_PATH = '/content/street_prior_Ntap32.npz'

sp_npz = np.load(PRIOR_PATH)
sp = {k: float(sp_npz[k]) for k in sp_npz.files}

print('\nR9 prior parameters')
for k, v in sp.items():
    print(f'  {k:20s} {v:.8g}')

print('\nImportant saved sanity numbers:')
print('  tap_p1_corr        =', sp.get('tap_p1_corr'))
print('  cf_corr_vs_numeric =', sp.get('cf_corr_vs_numeric'))

## 5. Read the real CFD and extract Fourier modes

We fit \(k=0,1,2,3\) **simultaneously** at the R9 shedding frequency.  
For a real field,

\[
q(t)=q_0 + 2\Re\left[\hat q_1e^{i\omega t}+\hat q_2e^{2i\omega t}+\cdots\right].
\]

The stored CFD coefficient below is therefore

\[
\hat q_k=\tfrac12(a_k-i b_k)
\]

when the least-squares basis is \(a_k\cos(k\omega t)+b_k\sin(k\omega t)\).

In [ ]:
import sys
sys.path.insert(0, '/content')
from text_flow import read_flow

Re, Ur, times, nodes_X, nodes_Y, Us, Vs, Ps = read_flow(LOCAL_DATA)
times = np.asarray(times)
x_all = nodes_X[0]
y_all = nodes_Y[0]

# Same reconstruction box used by the R9 evaluator.
in_box = ((x_all > -4.0) & (x_all < 8.0) &
          (y_all > -4.0) & (y_all < 4.0))
idx = np.where(in_box)[0]

x = x_all[idx].astype(float)
y = y_all[idx].astype(float)
U = Us[:, idx]
V = Vs[:, idx]
P = Ps[:, idx]

print('Re =', Re, '| Nt =', len(times), '| cropped nodes =', len(idx))

omega = sp['omega']
t = times - times[0]

# Simultaneous 0..3 harmonic least-squares fit to reduce leakage between harmonics.
cols = [np.ones_like(t)]
for k in range(1, 4):
    cols += [np.cos(k*omega*t), np.sin(k*omega*t)]
B = np.stack(cols, axis=1)

def fit_modes(F):
    coef, *_ = np.linalg.lstsq(B, F, rcond=None)
    modes = [coef[0].astype(complex)]
    j = 1
    for k in range(1, 4):
        modes.append(0.5*(coef[j] - 1j*coef[j+1]))
        j += 2
    return modes

u_modes = fit_modes(U)
v_modes = fit_modes(V)
p_modes = fit_modes(P)

u1, v1, p1 = u_modes[1], v_modes[1], p_modes[1]

# Free the large time-history slices after mode extraction.
del U, V, P, Us, Vs, Ps, nodes_X, nodes_Y

r = np.sqrt(x*x + y*y)
near_cyl = r < 0.75
near_wake = (~near_cyl) & (x >= 0.0) & (x < 3.0)
far_wake  = (~near_cyl) & (~near_wake) & (x >= 3.0)
wake_all  = (~near_cyl) & (x >= 0.0)

# A focused active-wake mask avoids letting thousands of nearly-zero outer nodes
# dominate the trust-reachability diagnostic.
qamp = np.sqrt(np.abs(u1)**2 + np.abs(v1)**2)
active_threshold = 0.10 * np.max(qamp[wake_all])
active_wake = wake_all & (qamp >= active_threshold)
active_far = far_wake & (qamp >= active_threshold)

regions = {
    'near-cylinder': near_cyl,
    'near-wake': near_wake,
    'far-wake': far_wake,
    'whole-wake': wake_all,
    'active-wake': active_wake,
    'active-far-wake': active_far,
}

print('omega = %.8f' % omega)
print('active-wake threshold = %.5g' % active_threshold)
for name, m in regions.items():
    print(f'{name:18s}: {m.sum():6d} nodes')

## 6. Reconstruct the exact closed-form R9 \(k=1\) prior on the CFD mesh

This reproduces the analytical part of `street_modes_k` in R9.

Important convention detail:

- the R9 network stores fluctuating modes in a one-sided convention with no factor 2 in time reconstruction;
- the CFD coefficients above are conventional Fourier coefficients with the \(+c.c.\) factor;
- therefore the **physical Fourier coefficient** of the prior is `amp_scale * cf_mode`;
- the actual R9 network quantity is twice that value.

In [ ]:
HA_RATIO = 0.281
NU = 1.0/100.0

def cf_modes_uv_np(x, y, prm, nk=1):
    G, Uc, xf, r0, phase, omega = (
        prm['Gamma'], prm['Uc'], prm['xf'], prm['r0'],
        prm['phase'], prm['omega'])
    ramp = prm.get('ramp', 0.75)
    delta = prm.get('delta', 0.35)

    a = 2*np.pi*Uc/omega
    h = HA_RATIO*a
    env = 0.5*(1 + np.tanh((x-xf)/ramp))
    rc2 = r0**2 + 4*NU*np.clip(x-xf, 0, None)/Uc

    us, vs = [], []
    for k in range(1, nk+1):
        att = np.exp(-(np.pi*k)**2 * rc2/a**2)
        tot_u = np.zeros_like(x, dtype=complex)
        tot_v = np.zeros_like(x, dtype=complex)

        for y_row, sgn_row, x0 in (
            (+h/2, -1.0, xf),
            (-h/2, +1.0, xf+a/2)
        ):
            yp = y-y_row
            sabs = np.sqrt(yp**2 + delta**2) - delta
            sgn = -np.tanh(yp/delta)
            Dk = np.exp(-2*np.pi*k*sabs/a)
            ph = -2*np.pi*k*(x-x0)/a - k*phase
            Ek = np.cos(ph) + 1j*np.sin(ph)
            base = sgn_row*G/(2*a) * Ek * Dk * att
            tot_u += sgn*base
            tot_v += 1j*base

        us.append(tot_u*env)
        vs.append(tot_v*env)
    return us, vs

us_cf, vs_cf = cf_modes_uv_np(x, y, sp, nk=1)

# Conventional physical Fourier coefficients: q = q0 + q1 e^(iwt) + c.c.
u1_prior = sp['amp_scale'] * us_cf[0]
v1_prior = sp['amp_scale'] * vs_cf[0]
p1_prior = -(1.0-sp['Uc']) * sp['scale_p'] * u1_prior

print('Prior fields generated on', len(x), 'CFD mesh nodes.')

## 7. Main test — prior alone vs CFD, then oracle scalar correction

The **raw prior** uses no full-field CFD information.

The **oracle-scaled prior** is allowed one complex number

\[
c_\mathrm{oracle}
=
\frac{\langle S,T\rangle}{\langle S,S\rangle},
\]

fitted to the real CFD \(k=1\) wake. This does **not** make a usable pressure-only model;
it diagnoses the error source:

- if oracle scaling helps a lot, R9 mostly has an amplitude/phase calibration problem;
- if it barely helps, the remaining error is mainly spatial-shape error.

In [ ]:
def complex_metrics(pred, truth, mask):
    p = np.asarray(pred)[mask]
    t = np.asarray(truth)[mask]
    n_p = np.linalg.norm(p)
    n_t = np.linalg.norm(t)
    rel = np.linalg.norm(p-t)/(n_t + 1e-30)
    corr = abs(np.vdot(p, t))/(n_p*n_t + 1e-30)
    mac = corr**2
    c = np.vdot(p, t)/(np.vdot(p, p) + 1e-30)
    return rel, corr, mac, c

def vector_oracle(uS, vS, uT, vT, mask):
    S = np.concatenate([uS[mask], vS[mask]])
    T = np.concatenate([uT[mask], vT[mask]])
    return np.vdot(S, T)/(np.vdot(S, S) + 1e-30)

rows = []
oracle_by_region = {}

for region, mask in regions.items():
    c_or = vector_oracle(u1_prior, v1_prior, u1, v1, mask)
    oracle_by_region[region] = c_or

    for field, S, T in [('u1', u1_prior, u1), ('v1', v1_prior, v1)]:
        raw_rel, raw_corr, raw_mac, _ = complex_metrics(S, T, mask)
        or_rel, or_corr, or_mac, _ = complex_metrics(c_or*S, T, mask)
        rows.append({
            'region': region,
            'field': field,
            'raw_rel_L2': raw_rel,
            'raw_corr': raw_corr,
            'raw_MAC': raw_mac,
            'oracle_rel_L2': or_rel,
            'oracle_corr': or_corr,
            'oracle_MAC': or_mac,
            'oracle_gain_abs': abs(c_or),
            'oracle_phase_deg': np.degrees(np.angle(c_or)),
        })

summary = pd.DataFrame(rows)
display(summary.round(4))

summary.to_csv(os.path.join(OUT_DIR, 'prior_vs_CFD_metrics.csv'), index=False)

c_wake = oracle_by_region['whole-wake']
print('\nWhole-wake common oracle scalar:')
print('  |c|   = %.4f' % abs(c_wake))
print('  phase = %.2f deg' % np.degrees(np.angle(c_wake)))
print('Saved:', os.path.join(OUT_DIR, 'prior_vs_CFD_metrics.csv'))

## 8. Visual comparison of CFD \(v_1\), raw R9 prior, and oracle-scaled prior

In [ ]:
import matplotlib.tri as mtri

tri = mtri.Triangulation(x, y)
mask_tri = np.any(r[tri.triangles] < 0.50, axis=1)
tri.set_mask(mask_tri)

def tripanel(field, title, filename, lim=None):
    fig, ax = plt.subplots(figsize=(10, 4.5))
    vals = np.real(field)
    if lim is None:
        lim = np.nanpercentile(np.abs(vals[wake_all]), 99)
    pc = ax.tripcolor(tri, vals, shading='gouraud', vmin=-lim, vmax=lim)
    circle = plt.Circle((0,0), 0.5, fill=False)
    ax.add_patch(circle)
    ax.set_xlim(-0.5, 8)
    ax.set_ylim(-2.5, 2.5)
    ax.set_aspect('equal')
    ax.set_xlabel('x/D')
    ax.set_ylabel('y/D')
    ax.set_title(title)
    fig.colorbar(pc, ax=ax, label='Re(v1)')
    fig.tight_layout()
    path = os.path.join(OUT_DIR, filename)
    fig.savefig(path, dpi=180)
    plt.show()
    return path

lim = np.nanpercentile(np.abs(np.real(v1)[wake_all]), 99)
tripanel(v1, 'CFD first harmonic: Re(v1)', 'v1_CFD.png', lim)
tripanel(v1_prior, 'R9 pressure-derived analytical prior: Re(v1)', 'v1_prior_raw.png', lim)
tripanel(c_wake*v1_prior, 'Oracle-scaled R9 prior: Re(v1)', 'v1_prior_oracle.png', lim)

## 9. Check the R9 pressure anchor at the actual 32 taps

This is separate from the saved `tap_p1_corr`, because that saved number comes from the
**image-vortex pressure-pattern matching step** used to choose phase/formation parameters.

Here we evaluate the **closed-form pressure anchor actually supplied to the R9 pressure network**:
\[
p_1^K \approx -(1-U_c)\,\mathrm{scale}_p\,u_1^K.
\]

In [ ]:
# Find cylinder mesh nodes and choose the same 32 evenly spaced requested tap positions.
eps = 1e-5
on_cyl = (r - 0.5)**2 < eps
xc_all, yc_all = x[on_cyl], y[on_cyl]
p1_cyl = p1[on_cyl]

theta_all = np.arctan2(yc_all, xc_all)
s_lin = np.linspace(0., 1., 32, endpoint=False)
xt = 0.5*np.cos(2*np.pi*s_lin)
yt = 0.5*np.sin(2*np.pi*s_lin)
pick = np.array([np.argmin((xc_all-xt[k])**2 + (yc_all-yt[k])**2) for k in range(32)])

x_tap, y_tap = xc_all[pick], yc_all[pick]
p1_tap = p1_cyl[pick]

us_tap, _ = cf_modes_uv_np(x_tap, y_tap, sp, nk=1)
u1_tap_prior = sp['amp_scale']*us_tap[0]
p1_tap_prior = -(1-sp['Uc'])*sp['scale_p']*u1_tap_prior

rel, corr, mac, c_tap = complex_metrics(
    p1_tap_prior, p1_tap, np.ones(32, dtype=bool)
)

print('Saved R9 image-street tap_p1_corr = %.4f' % sp.get('tap_p1_corr', np.nan))
print('Closed-form pressure anchor vs CFD taps:')
print('  rel L2              = %.4f' % rel)
print('  complex correlation = %.4f' % corr)
print('  MAC                  = %.4f' % mac)
print('  best scalar |c|      = %.4f' % abs(c_tap))
print('  best phase correction= %.2f deg' % np.degrees(np.angle(c_tap)))

theta_deg = (np.degrees(np.arctan2(y_tap, x_tap)) + 360) % 360
order = np.argsort(theta_deg)

fig, ax = plt.subplots(figsize=(9,4.5))
ax.plot(theta_deg[order], np.abs(p1_tap)[order], 'o-', label='CFD |p1|')
ax.plot(theta_deg[order], np.abs(p1_tap_prior)[order], 'o--', label='R9 closed-form prior |p1|')
ax.set_xlabel('tap angle [deg]')
ax.set_ylabel('|p1|')
ax.set_title('32-tap first-harmonic pressure amplitude')
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, 'tap_p1_amplitude.png'), dpi=180)
plt.show()

## 10. The key R9 trust-region test: can it reach **zero**, and can it reach **truth**?

R9 uses, for each complex fluctuating mode,

\[
q=S+A\,[\tanh(\Re N)+i\tanh(\Im N)],
\qquad A=\rho|S|+c.
\]

Therefore the allowed correction is a **square** in the complex plane:

\[
|\Re(q-S)|\le A,\qquad |\Im(q-S)|\le A.
\]

It is **not** a disk of radius \(A\).

For comparison, a true radial trust region would constrain

\[
|q-S|\le A.
\]

We evaluate both against the real CFD \(k=1\) mode.

In [ ]:
# R9 network convention: fluctuating mode is 2 x the conventional Fourier coefficient.
# For velocity, the actual R9 output is:
#   q = f_BC5 * (S + correction)
# while A = rho*|S| + cap is computed BEFORE f_BC5.
S_u = 2*u1_prior
S_v = 2*v1_prior
T_u = 2*u1
T_v = 2*v1

fbc5 = np.tanh(5.0*(r-0.5))

def square_zero_reachable(S, A):
    return (np.abs(S.real) <= A) & (np.abs(S.imag) <= A)

def square_truth_reachable(S, T, A, fmask):
    # Need T = fmask*(S + delta), with componentwise |delta| <= A.
    out = np.zeros(S.shape, dtype=bool)
    safe = np.abs(fmask) > 1e-8
    target_inner = np.zeros_like(S)
    target_inner[safe] = T[safe]/fmask[safe]
    d = target_inner-S
    out[safe] = (np.abs(d.real[safe]) <= A[safe]) & (np.abs(d.imag[safe]) <= A[safe])
    # On the cylinder f_BC5=0, only a zero target is reachable.
    out[~safe] = np.abs(T[~safe]) < 1e-6
    return out

def square_floor_output(S, A, fmask):
    rr = np.maximum(np.abs(S.real)-A, 0.0)
    ii = np.maximum(np.abs(S.imag)-A, 0.0)
    return np.abs(fmask)*np.sqrt(rr*rr + ii*ii)

def radial_zero_reachable(S, A):
    return np.abs(S) <= A

def radial_truth_reachable(S, T, A, fmask):
    out = np.zeros(S.shape, dtype=bool)
    safe = np.abs(fmask) > 1e-8
    target_inner = np.zeros_like(S)
    target_inner[safe] = T[safe]/fmask[safe]
    out[safe] = np.abs(target_inner[safe]-S[safe]) <= A[safe]
    out[~safe] = np.abs(T[~safe]) < 1e-6
    return out

def radial_floor_output(S, A, fmask):
    return np.abs(fmask)*np.maximum(np.abs(S)-A, 0.0)

def trust_stats(S, T, mask, rho, cap, geometry='square', fmask=fbc5):
    A = rho*np.abs(S) + cap
    if geometry == 'square':
        zr = square_zero_reachable(S, A)
        tr = square_truth_reachable(S, T, A, fmask)
        floor = square_floor_output(S, A, fmask)
    else:
        zr = radial_zero_reachable(S, A)
        tr = radial_truth_reachable(S, T, A, fmask)
        floor = radial_floor_output(S, A, fmask)

    true_amp = np.abs(T)
    denom = np.mean(true_amp[mask]) + 1e-30
    return {
        'zero_reachable_frac': np.mean(zr[mask]),
        'truth_reachable_frac': np.mean(tr[mask]),
        'mean_floor_over_mean_truth': np.mean(floor[mask])/denom,
        'median_floor_over_median_truth':
            np.median(floor[mask])/(np.median(true_amp[mask])+1e-30),
    }

rows = []
for geometry in ['square', 'radial']:
    for field, S, T in [('u1', S_u, T_u), ('v1', S_v, T_v)]:
        for region in ['near-wake', 'far-wake', 'active-wake', 'active-far-wake']:
            st = trust_stats(S, T, regions[region], rho=0.6, cap=0.12, geometry=geometry)
            rows.append({
                'geometry': geometry,
                'field': field,
                'region': region,
                'rho': 0.6,
                'cap': 0.12,
                **st
            })

trust_current = pd.DataFrame(rows)
display(trust_current.round(4))
trust_current.to_csv(os.path.join(OUT_DIR, 'trust_current_R9.csv'), index=False)

## 11. Screen \(\rho,c\) **without training**

This is the useful replacement for blindly launching a long R9b run.

For every candidate we measure, on the active far wake:

- **zero reachable fraction** — lower is better;
- **truth reachable fraction** — higher is better.

A good trust region should block the collapsed wake while still leaving the CFD answer inside,
or close to, the permitted correction set.

In [ ]:
rho_grid = np.array([0.10, 0.20, 0.25, 0.30, 0.40, 0.45, 0.60, 0.80])
cap_grid = np.array([0.00, 0.01, 0.02, 0.03, 0.05, 0.08, 0.12])

scan_rows = []
mask = active_far

for geometry in ['square', 'radial']:
    for rho_ in rho_grid:
        for cap_ in cap_grid:
            # Use v1 as the primary collapse diagnostic; it was the dominant R9 failure metric.
            st = trust_stats(S_v, T_v, mask, rho_, cap_, geometry)
            scan_rows.append({
                'geometry': geometry,
                'rho': rho_,
                'cap': cap_,
                **st
            })

scan = pd.DataFrame(scan_rows)
scan.to_csv(os.path.join(OUT_DIR, 'trust_parameter_scan.csv'), index=False)

# Ranking: first minimize dead-wake access, then maximize truth access.
ranked = scan.sort_values(
    ['geometry', 'zero_reachable_frac', 'truth_reachable_frac'],
    ascending=[True, True, False]
)
print('Top candidates by geometry (active far wake, v1):')
display(ranked.groupby('geometry').head(12).round(4))

def heatmap(metric, geometry, title, filename):
    sub = scan[scan.geometry == geometry]
    Z = np.full((len(cap_grid), len(rho_grid)), np.nan)
    for i, cap_ in enumerate(cap_grid):
        for j, rho_ in enumerate(rho_grid):
            Z[i,j] = sub[(sub.rho==rho_) & (sub.cap==cap_)][metric].iloc[0]
    fig, ax = plt.subplots(figsize=(9,5))
    im = ax.imshow(Z, origin='lower', aspect='auto')
    ax.set_xticks(range(len(rho_grid)), labels=[str(v) for v in rho_grid])
    ax.set_yticks(range(len(cap_grid)), labels=[str(v) for v in cap_grid])
    ax.set_xlabel('rho')
    ax.set_ylabel('cap')
    ax.set_title(title)
    fig.colorbar(im, ax=ax, label=metric)
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, filename), dpi=180)
    plt.show()

heatmap('zero_reachable_frac', 'square',
        'Current R9 square: fraction of active far wake where q1=0 is reachable',
        'square_zero_reachable.png')
heatmap('truth_reachable_frac', 'square',
        'Current R9 square: fraction of active far wake where CFD q1 is reachable',
        'square_truth_reachable.png')
heatmap('zero_reachable_frac', 'radial',
        'Radial trust region: fraction where q1=0 is reachable',
        'radial_zero_reachable.png')
heatmap('truth_reachable_frac', 'radial',
        'Radial trust region: fraction where CFD q1 is reachable',
        'radial_truth_reachable.png')

## 12. Soft anchored-prior collapse scan

Now test the alternative **loss-based** idea without training.

Along the path

\[
q_1(\alpha)=\alpha q_1^{CFD},\qquad 0\le\alpha\le1.5,
\]

evaluate

\[
L_K(\alpha)
=
\frac{\|q_1(\alpha)-S_1\|^2}{\|S_1\|^2}.
\]

If

\[
\left.\frac{dL_K}{d\alpha}\right|_{\alpha=0}<0,
\]

the prior actively pushes a dead \(k=1\) mode away from zero.

We compare the raw pressure-derived prior with the oracle-scaled prior.

In [ ]:
mask = active_wake
T = np.concatenate([u1[mask], v1[mask]])
S = np.concatenate([u1_prior[mask], v1_prior[mask]])
S_or = c_wake*S

alpha = np.linspace(0, 1.5, 301)

def anchored_curve(Sref):
    denom = np.linalg.norm(Sref)**2 + 1e-30
    return np.array([np.linalg.norm(a*T-Sref)**2/denom for a in alpha])

L_raw = anchored_curve(S)
L_or = anchored_curve(S_or)

dL0_raw = -2*np.real(np.vdot(T, S))/(np.linalg.norm(S)**2 + 1e-30)
dL0_or = -2*np.real(np.vdot(T, S_or))/(np.linalg.norm(S_or)**2 + 1e-30)

alpha_star_raw = np.real(np.vdot(T,S))/(np.vdot(T,T).real + 1e-30)
alpha_star_or  = np.real(np.vdot(T,S_or))/(np.vdot(T,T).real + 1e-30)

print('Raw pressure-derived prior:')
print('  dL/dalpha at 0 = %.6f' % dL0_raw)
print('  path minimum alpha = %.4f' % alpha_star_raw)
print('Oracle-scaled prior:')
print('  dL/dalpha at 0 = %.6f' % dL0_or)
print('  path minimum alpha = %.4f' % alpha_star_or)

fig, ax = plt.subplots(figsize=(9,5))
ax.plot(alpha, L_raw, label='anchored prior — pressure-derived')
ax.plot(alpha, L_or, label='anchored prior — oracle scalar')
ax.axvline(1.0, linestyle='--', label='true CFD amplitude alpha=1')
ax.set_xlabel('wake amplitude alpha')
ax.set_ylabel('normalized Karman prior loss')
ax.set_title('Does the prior push a collapsed k=1 wake away from zero?')
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, 'soft_prior_alpha_scan.png'), dpi=180)
plt.show()

## 13. Optional but valuable — compare \(k=1\) vorticity on a regular wake grid

The analytical prior is fundamentally a vortex-street model, so velocity correlation alone is
not the whole story.

We interpolate the CFD \(u_1,v_1\) onto a regular grid and compute

\[
\omega_1 = \frac{\partial v_1}{\partial x}
          -\frac{\partial u_1}{\partial y}.
\]

This is an **evaluation diagnostic only**. The raw CFD field is still never used to build the prior.

In [ ]:
from scipy.interpolate import griddata

gx = np.linspace(0.6, 8.0, 220)
gy = np.linspace(-2.0, 2.0, 120)
Xg, Yg = np.meshgrid(gx, gy)
pts = np.column_stack([x, y])

def interp_complex(values):
    re = griddata(pts, values.real, (Xg,Yg), method='linear')
    im = griddata(pts, values.imag, (Xg,Yg), method='linear')
    return re + 1j*im

u1g = interp_complex(u1)
v1g = interp_complex(v1)

# Analytical prior is evaluated directly on the same grid.
ug_cf, vg_cf = cf_modes_uv_np(Xg.ravel(), Yg.ravel(), sp, nk=1)
u1pg = (sp['amp_scale']*ug_cf[0]).reshape(Xg.shape)
v1pg = (sp['amp_scale']*vg_cf[0]).reshape(Xg.shape)

dx = gx[1]-gx[0]
dy = gy[1]-gy[0]

dvdx = np.gradient(v1g, dx, axis=1)
dudy = np.gradient(u1g, dy, axis=0)
w1g = dvdx-dudy

dvdx_p = np.gradient(v1pg, dx, axis=1)
dudy_p = np.gradient(u1pg, dy, axis=0)
w1pg = dvdx_p-dudy_p

valid = np.isfinite(w1g.real) & np.isfinite(w1g.imag)
W = w1g[valid]
Wp = w1pg[valid]

corr_w = abs(np.vdot(Wp,W))/(np.linalg.norm(Wp)*np.linalg.norm(W)+1e-30)
mac_w = corr_w**2
c_w = np.vdot(Wp,W)/(np.vdot(Wp,Wp)+1e-30)
err_w_raw = np.linalg.norm(Wp-W)/(np.linalg.norm(W)+1e-30)
err_w_or = np.linalg.norm(c_w*Wp-W)/(np.linalg.norm(W)+1e-30)

print('Vorticity k=1 diagnostic:')
print('  raw rel L2       = %.4f' % err_w_raw)
print('  complex corr     = %.4f' % corr_w)
print('  MAC              = %.4f' % mac_w)
print('  oracle rel L2    = %.4f' % err_w_or)
print('  oracle gain      = %.4f' % abs(c_w))
print('  oracle phase deg = %.2f' % np.degrees(np.angle(c_w)))

lim = np.nanpercentile(np.abs(np.real(w1g)), 99)

for Z, title, fname in [
    (w1g, 'CFD Re(omega1)', 'omega1_CFD.png'),
    (w1pg, 'R9 prior Re(omega1)', 'omega1_prior.png'),
    (c_w*w1pg, 'Oracle-scaled prior Re(omega1)', 'omega1_prior_oracle.png'),
]:
    fig, ax = plt.subplots(figsize=(10,4.5))
    im = ax.pcolormesh(Xg, Yg, np.real(Z), shading='auto', vmin=-lim, vmax=lim)
    ax.set_xlabel('x/D')
    ax.set_ylabel('y/D')
    ax.set_title(title)
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, fname), dpi=180)
    plt.show()

## 14. Final machine-readable verdict

This cell saves a compact JSON with the numbers I need to interpret the result with you.

After running the notebook, send me either:

- `R9_prior_diagnostic/verdict.json`, or
- just paste the printed verdict and the two trust-region tables.

No ModalPINN training is required for this notebook.

In [ ]:
def row_lookup(df, region, field):
    return df[(df.region==region) & (df.field==field)].iloc[0]

far_v = row_lookup(summary, 'far-wake', 'v1')
active_v = row_lookup(summary, 'active-far-wake', 'v1')

sq_now = trust_current[
    (trust_current.geometry=='square') &
    (trust_current.field=='v1') &
    (trust_current.region=='active-far-wake')
].iloc[0]

rad_now = trust_current[
    (trust_current.geometry=='radial') &
    (trust_current.field=='v1') &
    (trust_current.region=='active-far-wake')
].iloc[0]

verdict = {
    'prior_path': PRIOR_PATH,
    'omega': omega,
    'saved_tap_p1_corr': sp.get('tap_p1_corr'),
    'saved_cf_corr_vs_numeric': sp.get('cf_corr_vs_numeric'),
    'far_v1_raw_rel_L2': float(far_v.raw_rel_L2),
    'far_v1_raw_corr': float(far_v.raw_corr),
    'far_v1_oracle_rel_L2': float(far_v.oracle_rel_L2),
    'far_v1_oracle_gain_abs': float(far_v.oracle_gain_abs),
    'far_v1_oracle_phase_deg': float(far_v.oracle_phase_deg),
    'active_far_v1_raw_rel_L2': float(active_v.raw_rel_L2),
    'active_far_v1_raw_corr': float(active_v.raw_corr),
    'R9_square_zero_reachable_frac_active_far_v1': float(sq_now.zero_reachable_frac),
    'R9_square_truth_reachable_frac_active_far_v1': float(sq_now.truth_reachable_frac),
    'R9_radial_zero_reachable_frac_active_far_v1': float(rad_now.zero_reachable_frac),
    'R9_radial_truth_reachable_frac_active_far_v1': float(rad_now.truth_reachable_frac),
    'soft_prior_dL_dalpha_at_zero_raw': float(dL0_raw),
    'soft_prior_path_min_alpha_raw': float(alpha_star_raw),
}

# Vorticity values exist if the optional section ran.
for name in ['corr_w','mac_w','err_w_raw','err_w_or']:
    if name in globals():
        verdict[name] = float(globals()[name])

with open(os.path.join(OUT_DIR, 'verdict.json'), 'w') as f:
    json.dump(verdict, f, indent=2)

print(json.dumps(verdict, indent=2))
print('\nSaved all outputs to:', OUT_DIR)